# **entire rag pipeline**

## **import**

In [ ]:
import os
import json
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from dotenv import load_dotenv
load_dotenv()

### **we're gonna deal with two kinds of documents, first simple textual document and then a pdf**

In [ ]:
langchain_docs = []

### **1-text**

In [ ]:
## loading the documents

text_data_path = "docs"
if not os.path.exists(text_data_path):
    raise FileNotFoundError("no textual data found")
text_loader = DirectoryLoader(
    path=text_data_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
text_docs = text_loader.load()

##chunking

chunk_size = 1000
chunk_overlap = 0
textsplitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

text_chunks = textsplitter.split_documents(text_docs)
text_langchain_chunks = [
    Document(page_content=chunk.page_content,
              metadata={
                             "original_content":json.dumps(
                                 {
                                 "raw_text": chunk.page_content,
                                 
                             }
                             )
                         }
                     ) for chunk in text_chunks
             ] 

langchain_docs.append(text_langchain_chunks)


### **2-pdf**

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)


## loading the elements of the pdf
pdf_file_path = "./docs/attention-is-all-you-need_results.pdf"
pdf_elements = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["Image"],
    extract_image_block_to_payload=True
)

##chunking by title
pdf_chunks = chunk_by_title(
    pdf_elements,
    max_characters=3000,
    new_after_n_chars=2400,
    combine_text_under_n_chars=500
)

##seperating chunk content
pdf_chunks_with_content = []
for pdf_chunk in pdf_chunks:
    chunk_content = {
        "raw_text":pdf_chunks.text,
        "raw_tables":[],
        "raw_images":[]
    }
    if hasattr(pdf_chunk,"metadata") and hasattr(pdf_chunk.metadata,"origelements"):
        for ele in pdf_chunk.metadata.orig_elements:
            ele_type = type(ele).__name__
            if ele_type == "Table":
                html_table = getattr(ele,"text_as_html",ele.text)
                chunk_content["raw_tables"].append(html_table)
            elif ele_type == "Image":
                image_b64 = getattr(ele.metadata,"image_base64",ele.text)
                chunk_content["raw_images"].append(image_b64)
    pdf_chunks_with_content.append(chunk_content)


## adding ai summary for images and tables
for pdf_chunk in pdf_chunks_with_content:
    if pdf_chunk["raw_tables"] or pdf_chunk["raw_images"]:
        prompt = f"""
                You are creating a searchable description for document content retrieval.
        
                CONTENT TO ANALYZE:
                TEXT CONTENT: {chunk_content["raw_text"]}
        
                    """
        for i,table in enumerate(pdf_chunk["raw_tables"]):
             prompt += f"""
                table {i+1} : {table}

                """
        
        prompt += """
            YOUR TASK:
                            Generate a comprehensive, searchable description that covers:
            
                            1. Key facts, numbers, and data points from text and tables
                            2. Main topics and concepts discussed  
                            3. Questions this content could answer
                            4. Visual content analysis (charts, diagrams, patterns in images)
                            5. Alternative search terms users might use
            
                            Make it detailed and searchable - prioritize findability over brevity.
            
                            SEARCHABLE DESCRIPTION:
            """    
        final_prompt = [{
            "type":"text",
            "content":prompt
        }]
        for i,image in enumerate(pdf_chunk["raw_images"]):
            final_prompt.append({
                            "type":"image_url",
                            "image_url":{"url":f"data:image/jpeg;base64,{image}"}
                        })
        messages = [
            {"type":"user",
             "content":final_prompt
            }
        ]
        response = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=messages,
            temperature=0
        )
        pdf_chunk["summary"] = response.choices[0].message.content

        ## creating the langchain documents
pdf_langchain = []
for pdf_chunk in pdf_chunks_with_content:
    if pdf_chunk["summary"]:
        pdf_doc = Document(page_content=pdf_chunk["summary"],
                           metadata={
                                                        "original_content":json.dumps(
                                                            {
                                                            "raw_text": pdf_chunk["raw_text"],
                                                            "raw_tables":pdf_chunk["raw_tables"],
                                                            "raw_images":pdf_chunk["images"]
                                                            
                                                        }
                                                        )
                                                    })
        pdf_langchain.append(pdf_doc)
    else:
        pdf_langchain.append(Document(page_content=pdf_chunk["raw_text"],
                                metadata={
                                        "original_content":json.dumps(
                                         {
                                         "raw_text": pdf_chunk["raw_text"], })}))
langchain_docs.append(pdf_langchain)

     





    